In [ ]:

"""
# Transformer 端到端 demo —— 加载本地模型 + 推理打分

本 notebook 只负责**推理**: 加载训练侧脚本 `transformer_train.py` 产出的 `transformer_model.json`,
在平台注入的**测试集区间**上打分, **不训练**。

训练逻辑 (配置 / 模型结构 / 数据构建 / `train_and_save`) 全部沉淀在 `transformer_train.py` 里,
作为单一事实来源。流程拆成两个阶段:

1. **阶段一 (参赛者本地运行一次)** —— 在 `transformer_train.py` 所在目录执行
   `python transformer_train.py` (或在 notebook 里调用 `train_and_save(...)`),
   在写死的训练区间上从零训练, 把 **权重 + 标准化统计 + 结构超参** 存到 `transformer_model.json`。
2. **阶段二 (平台公榜阶段调用 `main`)** —— 直接加载 `transformer_model.json`, 在平台注入的测试区间上推理打分。

> 提交时请把 `transformer_train.py` 与训练好的 `transformer_model.json` 随本 notebook 一并上传。
> 公榜阶段平台只替换 `datasources / start_date / end_date` 并调用 `main`, 仅基于提交的权重做推理;
> 私榜阶段平台会用 `train_and_save` 在隔离环境从零重训, 故训练脚本需保持可运行、结果可复现。

推理复用训练侧的 `build_dataset`, 且标准化统计 (mean/std) 随权重一起存盘、推理时直接复用,
保证两阶段预处理严格一致, 杜绝数据泄漏与 train/infer 漂移。
"""

# ==== 从训练侧脚本导入共享定义 (配置 / 模型结构 / 数据构建 / 训练函数) ====
# transformer_train.py 是训练与推理的单一事实来源; 此处只复用, 不重复定义
import os

import numpy as np
import pandas as pd
from bigquant import dai
import torch
import structlog

from Transformer_modelsave_train import (
    MODEL_PATH, BATCH, USE_LOCAL_DATA,
    StockTransformer, build_dataset, pool, train_and_save, load_model,
)

logger = structlog.get_logger()


# ==== 加载权重做推理 (平台公榜阶段调用, 只替换 datasources/start_date/end_date) ====
def main(datasources, start_date, end_date):
    """加载已训练好的模型, 在样本外测试区间 (start_date~end_date) 上推理打分。

    本函数**不训练**: 权重来自随 notebook 上传的 MODEL_PATH 文件。
    start_date~end_date 为平台注入的【测试集区间】, 输出每日分数 ['date','instrument','score']。"""
    table = datasources["bar1m"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"未找到模型文件 {MODEL_PATH}; 请先运行 train_and_save(...) 训练并保存, 再随 notebook 一起上传")

    # 模型存为文本类文件 (JSON), 用 load_model 读回 (张量按 dtype/shape 还原)
    ckpt = load_model(MODEL_PATH, map_location=device)
    stats = (np.asarray(ckpt["mean"], np.float32), np.asarray(ckpt["std"], np.float32))
    model = StockTransformer(**ckpt["model_cfg"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    logger.info("已加载模型", path=MODEL_PATH, device=str(device))

    # ---------- 推理 (样本外测试区间) ----------
    logger.info("构建测试集并预测", start=str(start_date), end=str(end_date))
    Xte, _, idx_df, _ = build_dataset(table, start_date, end_date, "infer",
                                      pool(start_date, end_date), stats)
    preds = []
    Xte_t = torch.from_numpy(Xte)
    with torch.no_grad():
        for i in range(0, len(idx_df), BATCH):
            xb = Xte_t[i:i + BATCH].to(device)
            preds.append(model(xb).cpu().numpy())
    idx_df["score"] = np.concatenate(preds).astype(np.float64)

    # ---------- 对齐中证 1000 + 规范输出 ----------
    if USE_LOCAL_DATA:
        stk = idx_df[["date", "instrument"]].drop_duplicates()
    else:
        stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                        filters={"date": [start_date, end_date]}).df()
    result = (pd.merge(idx_df, stk, on=["date", "instrument"], how="inner")
                .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
                .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
                .reset_index(drop=True))
    logger.info("分数构建完成", rows=len(result), days=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result

# ==== 本地评估函数 (仅 USE_LOCAL_DATA=True 时使用) ====
# 压力时段定义 (来自比赛文档评分机制: 分 regime 评估稳健性)
STRESS_PERIODS = [
    ("压力期-1月", "2024-01-15", "2024-02-08"),
    ("压力期-9月", "2024-09-24", "2024-10-08"),
]


def _local_eval(factor_data, show=True):
    """本地评估: 计算 Rank IC、IR、多空10分组年化夏普比率(SR) 及压力期稳健性.

    对标比赛最终得分公式:
        Score_final = 0.25×Rank(IC_mean) + 0.25×Rank(IC_IR) + 0.25×Rank(SR) + 0.25×Rank(Stress)
    此处计算各分项指标的原始值, 供本地调试参考.
    """
    import matplotlib
    matplotlib.use("Agg")  # 无头环境不弹窗, 保存文件代替
    import matplotlib.pyplot as plt
    import pyarrow.dataset as ds
    import pyarrow.compute as pc

    sd = pd.to_datetime(factor_data["date"].min())
    ed = pd.to_datetime(factor_data["date"].max())
    instruments = factor_data["instrument"].unique().tolist()

    buf = (sd - pd.Timedelta(days=3)).strftime("%Y-%m-%d")
    sd_str, ed_str = sd.strftime("%Y-%m-%d"), ed.strftime("%Y-%m-%d")

    # 读取本地数据获取日收益率 (仅 close)
    filters = (ds.field("day") >= buf) & (ds.field("day") <= ed_str)
    filters = filters & ds.field("instrument").isin(instruments)
    _HERE = os.path.dirname(os.path.abspath(__file__))
    DATA_DIR = os.path.join(_HERE, "..", "..", "data")
    tbl = ds.dataset(str(DATA_DIR), format="parquet", partitioning="hive").to_table(
        columns=["date", "instrument", "close"], filter=filters)
    df = tbl.to_pandas()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    # 每日收盘价 -> 下一日收益
    df["day"] = df["date"].dt.normalize()
    daily = df.groupby(["day", "instrument"])["close"].last().reset_index()
    daily["ret"] = daily.groupby("instrument")["close"].pct_change().shift(-1)
    daily = daily.rename(columns={"day": "date"}).dropna(subset=["ret"])

    merged = pd.merge(factor_data, daily[["date", "instrument", "ret"]],
                      on=["date", "instrument"], how="inner").dropna(subset=["ret"])

    # ----- 辅助: 安全十分组 (重复边界时丢弃, 极端情况全归 Q1) -----
    def _qcut_10(s):
        try:
            return pd.qcut(s.rank(method="first"), 10, labels=range(1, 11), duplicates="drop")
        except ValueError:
            return pd.Series(np.full(len(s), 1, dtype=int), index=s.index)

    def _compute_ic_ls(df_seg):
        """给定一个截面数据子集 (含 date/score/ret), 返回 (ic_mean, ic_ir, sr)."""
        ic_series = df_seg.groupby("date").apply(
            lambda g: g["score"].corr(g["ret"], method="spearman"), include_groups=False)
        ic_m = ic_series.mean()
        ic_s = ic_series.std()
        ic_ir_val = ic_m / ic_s * np.sqrt(252) if ic_s > 1e-10 else 0.0
        # 十分组多空组合日收益 -> 年化 SR
        seg = df_seg.copy()
        seg["dg"] = seg.groupby("date")["score"].transform(_qcut_10)
        dls = (seg.groupby(["date", "dg"], observed=False)["ret"].mean()
               .reset_index().pivot(index="date", columns="dg", values="ret"))
        dls["ls"] = dls.get(10, 0) - dls.get(1, 0)
        sr_val = (dls["ls"].mean() / dls["ls"].std() * np.sqrt(252)) if dls["ls"].std() > 1e-10 else 0.0
        return ic_m, ic_ir_val, sr_val

    # ================================================================
    # 1) 全区间指标: IC_mean / IC_IR / SR
    # ================================================================
    ic_mean, ic_ir, sr = _compute_ic_ls(merged)
    logger.info("全区间评估", rank_ic_mean=round(ic_mean, 4),
                rank_ic_ir=round(ic_ir, 2), sr=round(sr, 2),
                sample_days=merged["date"].nunique())

    # 十分组平均收益 & 多空 Spread (仅展示)
    merged["group"] = merged.groupby("date")["score"].transform(_qcut_10)
    group_ret = merged.groupby("group")["ret"].mean()
    spread = group_ret.get(10, 0) - group_ret.get(1, 0)

    print("=" * 64)
    print("  全区间  ")
    print(f"    Rank IC Mean: {ic_mean:.4f}  |  IR: {ic_ir:.2f}  |  SR (年化): {sr:.2f}")
    print(f"    十分组收益: ", end="")
    for g in range(1, 11):
        r = group_ret.get(g, 0)
        print(f"Q{g}={r * 100:.4f}%  ", end="")
    print(f"|  LS Spread: {spread * 100:.4f}%")

    # ================================================================
    # 2) 压力时段 (Stress Regime) 评估
    # ================================================================
    merged["_d"] = pd.to_datetime(merged["date"])
    print("\n" + "-" * 64)
    print("  压力时段评估  ")
    stress_scores = []
    for label, s_start, s_end in STRESS_PERIODS:
        mask = (merged["_d"] >= s_start) & (merged["_d"] <= s_end)
        sub = merged[mask]
        if len(sub) < 10:
            print(f"    {label} ({s_start}~{s_end}): 样本不足, 跳过")
            continue
        s_ic_m, s_ic_ir, s_sr = _compute_ic_ls(sub)
        stress_scores.append({"label": label, "ic_mean": s_ic_m, "ic_ir": s_ic_ir, "sr": s_sr,
                              "days": sub["_d"].nunique(), "stocks": sub["instrument"].nunique()})
        print(f"    {label} ({s_start}~{s_end}):  IC={s_ic_m:.4f}  IR={s_ic_ir:.2f}  SR={s_sr:.2f}"
              f"  (交易日={sub['_d'].nunique()}, 股票={sub['instrument'].nunique()})")

    # 计算非压力期作为对照
    non_stress_mask = True
    for _, s_start, s_end in STRESS_PERIODS:
        non_stress_mask = non_stress_mask & ~((merged["_d"] >= s_start) & (merged["_d"] <= s_end))
    sub_normal = merged[non_stress_mask]
    if len(sub_normal) > 10:
        n_ic_m, n_ic_ir, n_sr = _compute_ic_ls(sub_normal)
        print(f"    非压力期:  IC={n_ic_m:.4f}  IR={n_ic_ir:.2f}  SR={n_sr:.2f}"
              f"  (交易日={sub_normal['_d'].nunique()})")

    # 压力综合得分: 各压力期 IC_mean 的平均值 (Stress 分项原始值)
    if stress_scores:
        avg_stress_ic = np.mean([s["ic_mean"] for s in stress_scores])
        avg_stress_ir = np.mean([s["ic_ir"] for s in stress_scores])
        avg_stress_sr = np.mean([s["sr"] for s in stress_scores])
        drop_ic = ic_mean - avg_stress_ic  # 压力期相对全区的 IC 衰减
        print(f"    压力综合(均值):  IC={avg_stress_ic:.4f}  IR={avg_stress_ir:.2f}  SR={avg_stress_sr:.2f}")
        print(f"    [诊断] 压力期 IC 衰减: {drop_ic:.4f}  (全区间 IC={ic_mean:.4f} - 压力平均 IC={avg_stress_ic:.4f})")
    print("=" * 64)

    # ================================================================
    # 3) 绘图: 十分组累积收益 + 压力期高亮
    # ================================================================
    if show:
        cum = (merged.groupby(["date", "group"])["ret"].mean()
               .reset_index().pivot(index="date", columns="group", values="ret")
               .fillna(0).cumsum())
        cum.columns = [f"Q{g}" for g in range(1, 11)]
        cum.index = pd.to_datetime(cum.index)
        ax = cum.plot(title="Cumulative Decile Returns (Local Eval)", figsize=(10, 5))
        ax.set_ylabel("Cumulative Return")
        # 高亮压力期背景
        for _, s_start, s_end in STRESS_PERIODS:
            ax.axvspan(pd.Timestamp(s_start), pd.Timestamp(s_end),
                       alpha=0.12, color="red", label=f"Stress: {s_start[:7]}")
        plt.tight_layout()
        fig_path = os.path.join(_HERE, "local_eval_plot.png")
        plt.savefig(fig_path, dpi=150)
        logger.info("评估图已保存", path=fig_path)
        plt.close()

    # 清理辅助列
    merged.drop(columns=["_d", "group"], inplace=True, errors="ignore")
    return merged


if __name__ == "__main__":
    # 只用 1 分钟 K 线作为输入数据
    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}

    # 本地首次运行: 若权重文件不存在, 先训练并保存 (上传前请确保已生成 transformer_model.json)
    # 正式做法是在终端执行 `python transformer_train.py` 训练一次, 这里仅为本地便捷兜底
    if not os.path.exists(MODEL_PATH):
        logger.info("未发现已保存模型, 开始训练", path=MODEL_PATH)
        train_and_save(datasources)

    # 本地用一小段区间模拟「平台注入的测试集区间」(训练区间已在 transformer_train.py 内写死)
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    logger.info("计算分数 (仅加载权重推理, 不重训)", start=start_date, end=end_date)
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())

    # 评估系统: 分数经风格剔除后等价于每日单因子, show=True 画绩效图 (IC / 分组 / 压力期)
    logger.info("开始评估分数")
    if USE_LOCAL_DATA:
        result = _local_eval(score_data, show=True)
    else:
        from bigmodule import M
        result = M.bigalpha_eval._latest(factor_data=score_data, show=True)
